# 003 Context Engineering

这是 LangChain Advanced usage 学习线的第三份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/context-engineering

学习目标：

1. 理解 context engineering 不是“把所有东西塞进 prompt”
2. 区分 model context、tool context、life-cycle context
3. 学会用 `dynamic_prompt` 根据 runtime 生成 prompt
4. 学会用 `wrap_model_call` 临时注入模型上下文
5. 学会用 `ToolRuntime.store` 让工具读写长期上下文
6. 学会按权限裁剪模型可见工具
7. 对比本仓库 Harness 的 context budget、ledger、subagent synthesis

这一讲使用 fake model，重点验证上下文进入 agent 的位置和生命周期。

## 1. Context Engineering 是什么

Context engineering 的核心不是“上下文越多越好”。

更准确的理解是：

```text
在正确的时间，把正确的信息，以正确的形式，放到正确的位置。
```

用 Java 来类比：

```text
Context engineering
  ~= Controller 入参 + RequestContext + Cache/Repository + Interceptor + DTO 裁剪
```

在 agent 系统里，常见上下文可以分三类：

| 类型 | 进入位置 | 例子 |
| --- | --- | --- |
| model context | 模型调用前 | system prompt、messages、可见 tools、response format |
| tool context | 工具执行时 | runtime.context、runtime.store、tool 参数 |
| life-cycle context | 多步运行期间 | 历史压缩、tool result 清理、长期记忆、ledger 摘要 |

本仓库 Harness 里对应的是：

- prompt / request_messages
- completed_steps / ledger
- context compact
- subagent result synthesis
- tool permission 和 approval 状态

In [2]:
from dataclasses import dataclass
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import (
    ClearToolUsesEdit,
    ContextEditingMiddleware,
    ModelRequest,
    SummarizationMiddleware,
    dynamic_prompt,
    wrap_model_call,
)
from langchain.tools import ToolRuntime, tool
from langchain_core.language_models.fake_chat_models import FakeListChatModel, FakeMessagesListChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.store.memory import InMemoryStore


class ToolCallingFakeModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self


def print_messages(result: dict) -> None:
    for message in result.get("messages", []):
        print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)


## 2. 定义请求上下文

先定义一个请求级 context。

它代表系统已经知道的外部事实，不需要模型从用户问题里猜。

In [3]:
@dataclass
class RequestContext:
    user_role: str
    data_scope: str
    doc_summary: str


context = RequestContext(
    user_role="Java 工程师",
    data_scope="public",
    doc_summary="Runtime 是 agent 执行时给 tool 和 middleware 的依赖注入入口。",
)

context


RequestContext(user_role='Java 工程师', data_scope='public', doc_summary='Runtime 是 agent 执行时给 tool 和 middleware 的依赖注入入口。')

## 3. Model Context：动态 system prompt

`dynamic_prompt` 可以根据 runtime 生成 system prompt。

这属于 model context：它影响模型如何理解任务，但不会写入长期记忆。

In [4]:
@dynamic_prompt
def prompt_from_context(request: ModelRequest) -> str:
    role = request.runtime.context.user_role
    print("dynamic prompt user_role:", role)
    return "你是教学助手。请用适合 " + role + " 的方式解释问题。"


prompt_agent = create_agent(
    model=FakeListChatModel(responses=["我会用 Java 类比解释 context engineering。"]),
    tools=[],
    context_schema=RequestContext,
    middleware=[prompt_from_context],
)

prompt_result = prompt_agent.invoke(
    {"messages": [{"role": "user", "content": "解释 context engineering"}]},
    context=context,
)

print_messages(prompt_result)


dynamic prompt user_role: Java 工程师
human 解释 context engineering
ai 我会用 Java 类比解释 context engineering。


## 4. Model Context：临时注入资料摘要

`wrap_model_call` 可以在模型调用前临时修改本次 request。

下面把 `runtime.context.doc_summary` 注入到模型 messages 里。

注意：这是临时 model context，不会写入 store，也不应该无限增长。

In [5]:
@wrap_model_call
def inject_doc_summary(request: ModelRequest, handler):
    extra_message = HumanMessage(content="临时资料摘要：" + request.runtime.context.doc_summary)
    request = request.override(messages=[*request.messages, extra_message])
    print("messages sent to model:", len(request.messages))
    return handler(request)


inject_agent = create_agent(
    model=FakeListChatModel(responses=["已基于临时资料摘要回答。"]),
    tools=[],
    context_schema=RequestContext,
    middleware=[inject_doc_summary],
)

inject_result = inject_agent.invoke(
    {"messages": [{"role": "user", "content": "Runtime 是什么？"}]},
    context=context,
)

print_messages(inject_result)


messages sent to model: 2
human Runtime 是什么？
ai 已基于临时资料摘要回答。


## 5. Model Context：按权限裁剪工具

模型可见的 tools 也是 context 的一部分。

如果用户只有 public 权限，就不应该让模型看到 private ledger 工具。

这和本仓库 Harness 的 tool permission / approval 是同一类问题。

In [9]:
@tool
def public_docs() -> str:
    """Search public docs."""
    return "public docs"


@tool
def private_ledger() -> str:
    """Search private ledger."""
    return "private ledger"


@wrap_model_call
def filter_tools_by_scope(request: ModelRequest, handler):
    before = [getattr(tool_item, "name", str(tool_item)) for tool_item in request.tools]

    if request.runtime.context.data_scope == "public":
        request = request.override(
            tools=[tool_item for tool_item in request.tools if getattr(tool_item, "name", "") == "public_docs"]
        )

    after = [getattr(tool_item, "name", str(tool_item)) for tool_item in request.tools]
    print("tools before:", before)
    print("tools after:", after)
    return handler(request)


tool_filter_agent = create_agent(
    model=ToolCallingFakeModel(responses=[AIMessage(content="模型只看到了允许的工具。")]),
    tools=[public_docs, private_ledger],
    context_schema=RequestContext,
    middleware=[filter_tools_by_scope],
)

tool_filter_result = tool_filter_agent.invoke(
    {"messages": [{"role": "user", "content": "查一些资料"}]},
    context=context,
)

print_messages(tool_filter_result)


tools before: ['public_docs', 'private_ledger']
tools after: ['public_docs']
human 查一些资料
ai 模型只看到了允许的工具。


## 6. Tool Context：工具读写长期上下文

tool context 是工具执行时能拿到的信息，例如：

- `runtime.context`
- `runtime.store`
- tool 参数
- tool_call_id

下面让工具把一个学习偏好写入 store。

In [7]:
store = InMemoryStore()


@tool
def save_learning_preference(key: str, value: str, runtime: ToolRuntime[RequestContext]) -> str:
    """Save a learning preference."""
    runtime.store.put(("learning", runtime.context.user_role), key, {"value": value})
    return "saved"


preference_model = ToolCallingFakeModel(
    responses=[
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "save_learning_preference",
                    "args": {"key": "style", "value": "多用 Java 类比"},
                    "id": "call_1",
                }
            ],
        ),
        AIMessage(content="偏好已保存。"),
    ]
)

preference_agent = create_agent(
    model=preference_model,
    tools=[save_learning_preference],
    context_schema=RequestContext,
    store=store,
)

preference_result = preference_agent.invoke(
    {"messages": [{"role": "user", "content": "记住我的学习偏好"}]},
    context=context,
)

print_messages(preference_result)
print("store direct read:", store.get(("learning", "Java 工程师"), "style").value)


human 记住我的学习偏好
ai 
tool_calls: [{'name': 'save_learning_preference', 'args': {'key': 'style', 'value': '多用 Java 类比'}, 'id': 'call_1', 'type': 'tool_call'}]
tool saved
ai 偏好已保存。
store direct read: {'value': '多用 Java 类比'}


## 7. Life-cycle Context：历史压缩和工具结果清理

agent 跑久了会遇到两个问题：

1. messages 越来越长
2. tool result 越来越多，挤占模型上下文

LangChain 提供 built-in middleware 来处理这类生命周期上下文：

- `SummarizationMiddleware`：把旧消息压缩成摘要
- `ContextEditingMiddleware` + `ClearToolUsesEdit`：清理旧 tool use / tool result

这一格只实例化它们，避免引入额外模型调用。

In [8]:
summarization = SummarizationMiddleware(
    model=FakeListChatModel(responses=["摘要：用户正在学习 context engineering。"]),
    trigger=("messages", 12),
    keep=("messages", 6),
)

context_editing = ContextEditingMiddleware(
    edits=[ClearToolUsesEdit(trigger=20, keep=3, clear_tool_inputs=True)]
)

print(type(summarization).__name__)
print(type(context_editing).__name__)


SummarizationMiddleware
ContextEditingMiddleware


## 8. 不同 Context 的生命周期

| Context | 位置 | 生命周期 | 适合内容 |
| --- | --- | --- | --- |
| system prompt | model context | 单次或多次模型调用 | 角色、规则、回答风格 |
| messages | model/state context | 本次 agent 运行中增长 | 用户问题、模型回答、tool result |
| runtime.context | runtime context | 单次 invocation | user_id、locale、权限范围 |
| runtime.store | persistent context | 跨 invocation | 用户偏好、长期记忆 |
| injected context | transient model context | 单次模型调用 | 当前资料摘要、检索片段 |
| visible tools | model context | 单次模型调用 | 当前允许模型调用的工具 |

工程判断：不要把持久信息混进临时 prompt，也不要把临时检索结果写进长期记忆。

## 9. 和本仓库 Harness 的对应关系

| LangChain Context Engineering | 本仓库 Harness 对应点 |
| --- | --- |
| dynamic prompt | planner/system prompt 组装 |
| request.messages | `request_messages` |
| runtime.context | session/request 级上下文 |
| runtime.store | 长期记忆 / DB / 缓存 |
| tool filtering | ToolRegistry + permission / approval |
| summarization | `_compact_history(...)` / completed_steps 压缩 |
| tool result cleanup | ledger 裁剪 / output eviction |
| subagent synthesis | 把局部观察压缩成主流程可执行上下文 |

最重要的判断：

```text
上下文管理不是一次性拼 prompt，而是贯穿 agent 生命周期的工程控制。
```

## 10. 本讲练习

请判断下面信息应该放在哪里：

1. 当前用户只能看 public 数据
2. 用户上传文档的 200 字摘要
3. 用户长期偏好：喜欢 Java 类比
4. 过去 30 轮对话的压缩摘要
5. 当前模型允许看到哪些工具

参考答案：

1. `runtime.context.data_scope`
2. 临时 model context，可以用 `wrap_model_call` 注入
3. `runtime.store`
4. life-cycle context，可以用 summarization 管理
5. model context，可以在 `wrap_model_call` 里裁剪 `request.tools`

## 11. 本讲小结

这一讲的核心：

```text
Context engineering 是上下文的分层、裁剪、注入、持久化和生命周期治理。
```

你现在应该能判断：

- 哪些信息应该进 prompt
- 哪些信息应该放 runtime.context
- 哪些信息应该放 store
- 哪些信息只适合临时注入
- 哪些上下文需要压缩或清理

下一步可以继续学习 LangChain 的 streaming，或者把本仓库 Harness 的 context compact 设计迁移成 LangChain middleware。